In [ ]:
!pip install roboflow ultralytics python-dotenv

In [ ]:
!pip install tensorflow==2.21.0 keras==3.14.1

In [ ]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np
import tensorflow
from tensorflow.keras.layers import Flatten, Dense, ReLU, Activation

load_dotenv()  # reads .env from repo root

def get_secret(key):
    val = os.getenv(key)
    if val:
        return val
    try:
        from google.colab import userdata
        return userdata.get(key)
    except Exception:
        raise ValueError(f"No value for '{key}' in .env or Colab secrets")

In [ ]:
REPO_ROOT = Path(".").resolve()

additional_dataset_directory_path = str(REPO_ROOT / "datasets/additional")
synthetic_dataset_directory_path   = str(REPO_ROOT / "datasets/synthetic")

# cropped ad images with Label Studio keypoint annotations
dataset_cropped_directory_path = str(REPO_ROOT / "datasets/cropped")
cropped_json = dataset_cropped_directory_path + "/project-2-at-2026-06-07-19-37-c29d30b5.json"

YOLO_MODEL_PATH  = str(REPO_ROOT / "models/bounding_best.pt")
KERAS_MODEL_PATH = str(REPO_ROOT / "models/best_point.keras")

---

In [ ]:
RETRAIN = True  # set False to skip training and load saved model
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16
VAL_SPLIT  = 0.2

In [ ]:
def parse_ls_export(json_path, images_dir):
    with open(json_path) as f:
        tasks = json.load(f)

    image_paths, labels = [], []

    for task in tasks:
        filename = Path(task["image"]).name
        path = str(Path(images_dir) / filename)

        points = {}
        for kp in task["keypoints"]:
            label = kp["keypointlabels"][0]
            points[label] = (kp["x"] / 100.0, kp["y"] / 100.0)

        try:
            row = [
                points["tl"][0], points["tl"][1],
                points["tr"][0], points["tr"][1],
                points["br"][0], points["br"][1],
                points["bl"][0], points["bl"][1],
            ]
        except KeyError:
            continue

        image_paths.append(path)
        labels.append(row)

    return image_paths, np.array(labels, dtype=np.float32)

In [ ]:
image_paths, labels = parse_ls_export(cropped_json, dataset_cropped_directory_path)
print(f"{len(image_paths)} labeled images")

In [ ]:
def load_and_preprocess(path, label):
    img = tensorflow.io.read_file(path)
    img = tensorflow.image.decode_jpeg(img, channels=3)
    img = tensorflow.image.resize(img, IMG_SIZE)
    img = tensorflow.keras.applications.resnet50.preprocess_input(img)
    return img, label

n = len(image_paths)
indices = np.random.permutation(n)
val_idx, train_idx = indices[:int(n*VAL_SPLIT)], indices[int(n*VAL_SPLIT):]

def make_ds(idx, shuffle=False):
    ds = tensorflow.data.Dataset.from_tensor_slices(
        ([image_paths[i] for i in idx], labels[idx])
    )
    ds = ds.map(load_and_preprocess, num_parallel_calls=tensorflow.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(512)
    return ds.batch(BATCH_SIZE).prefetch(tensorflow.data.AUTOTUNE)

In [ ]:
train_ds = make_ds(train_idx, shuffle=True)
val_ds = make_ds(val_idx)

In [ ]:
imagenet_base = tensorflow.keras.applications.ResNet50(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
imagenet_base.trainable = False

In [ ]:
model = tensorflow.keras.Sequential([
    imagenet_base,
    Flatten(),
    Dense(256),
    ReLU(),
    Dense(8),
    # Activation("sigmoid"),
])

In [ ]:
model.compile(optimizer=tensorflow.keras.optimizers.Adam(1e-4), loss=tensorflow.keras.losses.Huber())
model.summary()

In [ ]:
callbacks = [
    tensorflow.keras.callbacks.ModelCheckpoint("best.keras", monitor="val_loss", save_best_only=True),
    tensorflow.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tensorflow.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
]

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

In [ ]:
model.get_layer("resnet50").trainable = True
model.compile(optimizer=tensorflow.keras.optimizers.Adam(1e-5), loss=tensorflow.keras.losses.Huber())

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks)

In [ ]:
if RETRAIN:
    model.save(KERAS_MODEL_PATH)

In [ ]:
if not RETRAIN:
    model = tensorflow.keras.models.load_model(KERAS_MODEL_PATH)

In [ ]:
import cv2
from PIL import Image
import random

img_path = random.choice(image_paths)

# preprocess
img = tensorflow.keras.utils.load_img(img_path, target_size=IMG_SIZE)
img = tensorflow.keras.utils.img_to_array(img)
img = tensorflow.keras.applications.resnet50.preprocess_input(img)
img = tensorflow.expand_dims(img, axis=0)

# predict + clip
pred = model.predict(img)
pred = np.clip(pred, 0, 1)

# visualize on original (unprocessed) image
orig = tensorflow.keras.utils.load_img(img_path)
orig = tensorflow.keras.utils.img_to_array(orig).astype(np.uint8).copy()
h, w = orig.shape[:2]

tl_x, tl_y, tr_x, tr_y, br_x, br_y, bl_x, bl_y = pred[0]

points = {
    "tl": (int(tl_x * w), int(tl_y * h)),
    "tr": (int(tr_x * w), int(tr_y * h)),
    "br": (int(br_x * w), int(br_y * h)),
    "bl": (int(bl_x * w), int(bl_y * h)),
}

pts = [points["tl"], points["tr"], points["br"], points["bl"]]
for i in range(4):
    cv2.line(orig, pts[i], pts[(i+1) % 4], (0, 255, 0), 2)

for label, (x, y) in points.items():
    cv2.circle(orig, (x, y), 5, (0, 0, 255), -1)
    cv2.putText(orig, label, (x+5, y-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,0), 1)

display(Image.fromarray(orig))

In [ ]:
import tensorflow as tf
import keras

print("Your Local TensorFlow:", tf.__version__)
print("Your Local Keras:", keras.__version__)